In [1]:
from pystac_client import Client
from odc.stac import load
import depal_ck as dep

**Geometric Median and Absolute Deviations (GeoMAD)** product is an cloud-free annual mosaic that uses a more robust method of determining the median observation than a simple median.

Along with the median observation, the GeoMAD produces three measures of variance, or absolute deviations, which helps to understand how the data over the time period changes. For example, some areas, such as desert, will change very little. Whereas crop land will change more. All of these values are useful in understand what is happening in the area covered by the GeoMAD.


In [2]:
client = Client.open(
    "https://stac.staging.digitalearthpacific.org"
)

aoi = dep.get_country_admin_boundary("Cook Islands", "Island Council", "Rarotonga")
bbox = dep.get_bbox(aoi)

items = client.search(
    bbox = bbox,
    datetime = "2022",
    collections = ["dep_s2_geomad"],
).item_collection()

print(f"Found {len(items)} items")

Found 1 items


In [3]:
data = load(
    items,
    bbox=bbox,
    measurements=["B04", "B03", "B02"],
    chunks={"x": 2048, "y": 2048},
    resolution=10,
)

data = data.rename_vars({"B04": "red", "B03": "green", "B02": "blue"})
data

<xarray.Dataset> Size: 13MB
Dimensions:      (y: 889, x: 1209, time: 1)
Coordinates:
  * y            (y) float64 7kB -2.4e+06 -2.4e+06 ... -2.409e+06 -2.409e+06
  * x            (x) float64 10kB 5.585e+06 5.585e+06 ... 5.597e+06 5.597e+06
    spatial_ref  int32 4B 3832
  * time         (time) datetime64[ns] 8B 2022-01-01
Data variables:
    red          (time, y, x) float32 4MB dask.array<chunksize=(1, 889, 1209), meta=np.ndarray>
    green        (time, y, x) float32 4MB dask.array<chunksize=(1, 889, 1209), meta=np.ndarray>
    blue         (time, y, x) float32 4MB dask.array<chunksize=(1, 889, 1209), meta=np.ndarray>

In [4]:
data.odc.explore(vmin=0, vmax=2000)